In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vipullrathod/fish-market/Fish.csv


In [2]:
from pathlib import Path

csv_files= list(Path('/kaggle/input').rglob('*.csv'))
csv_files

[PosixPath('/kaggle/input/datasets/vipullrathod/fish-market/Fish.csv')]

In [3]:
csv_path = csv_files[0]
df = pd.read_csv(csv_path)
df.head()

,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340


In [4]:
display('[shape]',df.shape)
print()
display('[head]', df.head())
print()
display('[info]',df.info())
print()
display('[isna]',df.isna().sum())
print()
display('[des]',df.describe())

'[shape]'

(159, 7)

'[head]'

,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Species  159 non-null    object 
 1   Weight   159 non-null    float64
 2   Length1  159 non-null    float64
 3   Length2  159 non-null    float64
 4   Length3  159 non-null    float64
 5   Height   159 non-null    float64
 6   Width    159 non-null    float64
dtypes: float64(6), object(1)
memory usage: 8.8+ KB


...

None

'[isna]'

Species    0
Weight     0
Length1    0
Length2    0
Length3    0
Height     0
Width      0
dtype: int64

'[des]'

,Weight,Length1,Length2,Length3,Height,Width
count,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000
mean,398.326415,26.247170,28.415723,31.227044,8.970994,4.417486
std,357.978317,9.996441,10.716328,11.610246,4.286208,1.685804
min,0.000000,7.500000,8.400000,8.800000,1.728400,1.047600
25%,120.000000,19.050000,21.000000,23.150000,5.944800,3.385650
50%,273.000000,25.200000,27.300000,29.400000,7.786000,4.248500
75%,650.000000,32.700000,35.500000,39.650000,12.365900,5.584500
max,1650.000000,59.000000,63.400000,68.000000,18.957000,8.142000


In [5]:
# print(df.columns[2:])
# print(df.columns[1])
# X = df[df.columns[2:]]
# y = df[df.columns[1]]

# 열의 순서가 변경될 수 있어서 고정.
feature_cols = [
    'Length1', 'Length2', 'Length3',
    'Height', 'Width'
]

X = df[feature_cols]
y = df['Weight']

display(X.shape)
display(y.shape)

(159, 5)

(159,)

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state =42
)
display(X_train.shape)
display(X_test.shape)
display(y_train.shape)
display(y_test.shape)

(127, 5)

(32, 5)

(127,)

(32,)

In [7]:
import numpy as np

baseline_pred= np.full(
    shape = len(y_test),
    fill_value = y_train.mean()
)
print(baseline_pred[:5])

[386.79448819 386.79448819 386.79448819 386.79448819 386.79448819]


In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

ridge_model = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha =1.0)),
])

ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)
print('finish')

finish


In [9]:
from sklearn.metrics import(
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test,ridge_pred)
rmse = mean_squared_error(
    y_test,ridge_pred
)**0.5

r2 = r2_score(y_test,ridge_pred)
print(f'MAE: {mae:.2f} g')
print(f'RMSE: {rmse:.2f} g')
print(f'R²: {r2:.4f}')

MAE: 104.89 g
RMSE: 132.51 g
R²: 0.8766


In [10]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()

linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

print('finish')

finish


In [11]:
def regression_metrics(model_name, y_true, y_pred):
    return{
        'Model': model_name,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'R2': r2_score(y_true, y_pred)   
    }

result =pd.DataFrame([
    regression_metrics('baseline',y_test, baseline_pred),
    regression_metrics('Ridge',y_test, ridge_pred),
    regression_metrics('linearRegression',y_test, linear_pred),
    
])
result

,Model,MAE,RMSE,R2
0,baseline,329.768406,381.474104,-0.023082
1,Ridge,104.886242,132.510134,0.876554
2,linearRegression,103.909417,129.475431,0.882143


1. 이 문제는 왜 회귀인가?    
  A]범주형의 값을 분류해서 판단하는게 아니라 연속된 실수 값으로 물고기의 무게를 예측하기 때문 
2. X와 y는 각각 무엇인가?
  A] X는 특성을 말하며 (개수, 특성의 수)로 2차원 배열, y는 어떤 물고기인지 정답을 맞추기위한 무게값임
3. Ridge 전에 StandardScaler를 사용한 이유는 무엇인가?  
  A] 사용하는 범위가 다르기때문, 단위가 다르더라도 연속형의 값이 비슷한 범위라면 할 필요없음.\
    >>정답:Ridge는 계수의 제곱에 벌점을 주기 때문에
    >>특성의 크기가 다르면 계수에 적용되는 규제가 불공정해질 수 있다.
    >>따라서 StandardScaler로 각 특성을 평균 0, 표준편차 1 수준으로 맞춘다.
5. LinearRegression과 Ridge 중 test RMSE가 더 작은 모델은 무엇인가?
  A] 선형회귀가 Ridge보다 작아. 129.47이고 릿지는 132.51이잖아. 작을수록 좋은 모델이니까 오차기준으로는 선형회귀모델을 추천해.
6. 현재 결과만으로 어느 모델이 항상 더 좋다고 단정할 수 없는 이유는 무엇인가?
  A] 해당 결과는 오차값만 확인했지 정답률 이런건 확인하지않아서 단정하지못해.
     >>정답: 데이터가 159개로 적고 한 번 나눈 32개의 test set에서만 평가했기 때문에,
     >>데이터 분할이 달라지면 결과도 달라질 수 있다.
     >>교차검증 결과까지 확인해야 어느 모델이 더 안정적으로 좋은지 판단할 수 있다